# Cine de los 90 en el catálogo de Netflix**Escenario:** una productora especializada en estilos nostálgicos desea investigar laspelículas estrenadas durante la década de 1990.**Objetivo:** determinar la duración más frecuente de esas películas y cuántas de laspertenecientes al género de acción no superan los 90 minutos.## Los datos### `netflix_data.csv`| Columna | Descripción ||---|---|| `show_id` | ID del título || `type` | Tipo de título (película o serie) || `title` | Título || `director` | Director o directora || `cast` | Reparto || `country` | País de origen || `date_added` | Fecha de incorporación a Netflix || `release_year` | Año de estreno || `duration` | Duración en minutos (películas) o temporadas (series) || `description` | Descripción del título || `genre` | Género del título |

## 1. Carga de datos

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltnetflix_df = pd.read_csv("netflix_data.csv")print(f"Registros totales: {len(netflix_df)}")netflix_df.head()

## 2. Filtrado del subconjuntoSe seleccionan las películas estrenadas entre 1990 y 1999.El filtro por `type == 'Movie'` no es opcional: en las series la columna `duration` expresanúmero de temporadas, de modo que incluirlas contaminaría cualquier cálculo sobre esa variable.

In [ ]:
filtered_netflix = netflix_df.query("1990 <= release_year < 2000 and type == 'Movie'")print(f"Películas de los 90: {len(filtered_netflix)}")

## 3. Duración más frecuente`mode()` devuelve una Series, ya que puede existir más de un valor igualmente frecuente.Se toma el primer elemento y se convierte a entero.El valor se extrae del cálculo y no se introduce manualmente: de este modo el notebook siguesiendo correcto si el conjunto de datos se actualiza.

In [ ]:
duration = int(filtered_netflix['duration'].mode()[0])print(f"Duración más frecuente: {duration} minutos")

### Comparación con otras medidas de posiciónLa moda por sí sola puede inducir a error. Conviene contrastarla con la mediana y la mediapara comprobar si la distribución es simétrica.

In [ ]:
print(filtered_netflix['duration'].describe().round(1))print(f"\nModa:    {duration}")print(f"Mediana: {filtered_netflix['duration'].median():.0f}")print(f"Media:   {filtered_netflix['duration'].mean():.1f}")

La moda (94) es inferior a la mediana (108), que a su vez es inferior a la media (115,1).Esa relación indica una distribución **sesgada hacia la derecha**: existe una cola de películaslargas que desplaza la media al alza sin representar al grueso del conjunto.

## 4. Películas de acción de 90 minutos o menos`action_movies` se construye a partir de `filtered_netflix`, que ya contiene el filtro dedécada y tipo, en lugar de repetir la consulta sobre el DataFrame completo.El recuento se resuelve mediante una máscara booleana. La expresión `duration <= 90` genera unaSeries de valores `True`/`False` y `sum()` los suma tratando `True` como 1. Es equivalente arecorrer la columna con un bucle, pero se evalúa sobre la columna completa en una solaoperación.

In [ ]:
action_movies = filtered_netflix.query("genre == 'Action'")short_movie_count = (action_movies['duration'] <= 90).sum()print(f"Películas de acción en los 90: {len(action_movies)}")print(f"De 90 minutos o menos: {short_movie_count}")print(f"Porcentaje: {short_movie_count / len(action_movies):.1%}")

## 5. Análisis adicional: duración por géneroLas medidas globales ocultan diferencias relevantes. Se calcula la duración media de losgéneros con mayor representación en la muestra.

In [ ]:
top_genres = filtered_netflix['genre'].value_counts().head(6).indexduracion_genero = (    filtered_netflix[filtered_netflix['genre'].isin(top_genres)]    .groupby('genre')['duration']    .agg(media='mean', titulos='count')    .round(1)    .sort_values('media'))duracion_genero

Entre el género de menor y mayor duración media median 79 minutos. La duración no es unacaracterística homogénea de la década, sino que depende del género.

## 6. Visualización

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))# Distribución de duracionesax1.hist(filtered_netflix['duration'], bins=25, color='#4C72B0', edgecolor='white')ax1.axvline(duration, color='#C44E52', linestyle='--', linewidth=2,            label=f'Moda: {duration} min')ax1.axvline(filtered_netflix['duration'].mean(), color='#55A868', linestyle='--',            linewidth=2, label=f"Media: {filtered_netflix['duration'].mean():.0f} min")ax1.set_title('Distribución de duraciones (1990-1999)')ax1.set_xlabel('Duración (minutos)')ax1.set_ylabel('Nº de películas')ax1.legend()# Duración media por géneroax2.barh(duracion_genero.index, duracion_genero['media'], color='#4C72B0')ax2.axvline(90, color='#C44E52', linestyle='--', linewidth=1.5, label='90 min')ax2.set_title('Duración media por género')ax2.set_xlabel('Duración media (minutos)')ax2.legend()plt.tight_layout()plt.savefig('portada.png', dpi=150, bbox_inches='tight')plt.show()

## 7. Conclusiones1. **La duración más frecuente es de 94 minutos**, si bien la media asciende a 115,1. La   distribución está sesgada hacia la derecha, de modo que ambas medidas responden a preguntas   distintas.2. **Solo 7 de las 48 películas de acción** de la década duran 90 minutos o menos, un 14,6 %.   El género promedia 120,1 minutos: su ritmo acelerado se manifiesta en la cadencia interna de   las escenas y no en un metraje reducido.3. **La duración depende del género antes que de la década.** La recomendación para una   producción de estilo noventero es tomar la referencia del género correspondiente y no la   moda global.### LimitacionesEl conjunto analizado son las 183 películas de los noventa que Netflix mantiene licenciadashoy, no la producción cinematográfica de aquel periodo. Asimismo, los datos no recogen ningunavariable de audiencia o retención, por lo que la relación entre metraje y atención del públicoqueda planteada como hipótesis y no como resultado.